# Gold V2 — Google Colab Runner

Build **Gold V2 from an already published, complete Silver V2 snapshot**. No Bronze ingestion, Silver V2 republish, training, or Gold V1 rewriting.

Run **all cells in order**. The notebook clones `VTC_GIT_REF` (default `main`; for review use `feat/gold-v2-pipeline-colab-runner`), mounts the shared Drive, verifies Silver V2 + Silver V1 lineage and all Silver V2 auxiliary artifacts, then optionally publishes Gold V2.

**Safe by default:** `PUBLISH_GOLD_V2=False`. Set it to `True` only after reviewing the selected source report and inspecting the Silver V2 cleaning audit. Partial output or stale `.uploading` files fail closed. **Do not delete or overwrite files automatically.**


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_gold_v2_project")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True,
    )
    if dirty.strip():
        raise RuntimeError("Colab checkout has uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project directory contains non-Git files")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=PROJECT_DIR, check=True,
)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing project Python sources: {SRC_DIR}")
sys.path[:] = [part for part in sys.path if part != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))
prefix = "vietnamese_topic_classification"
for name in sorted(
    [name for name in sys.modules if name == prefix or name.startswith(prefix + ".")],
    key=len, reverse=True,
):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader, GoldV2Pipeline
from vietnamese_topic_classification.data.silver import SilverReader

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True,
).strip()
print("Git source:", BRANCH, "@", revision)


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Mounted Data Lake:", DATA_LAKE_ROOT)


In [ ]:
# Supply an explicit filename if more than one Silver V2 quality report exists.
SILVER_V2_REPORT_NAME = None
reports_dir = DATA_LAKE_ROOT / "04_reports" / "silver" / "v2"
available = sorted(path.name for path in reports_dir.glob("silver_quality_v2_*.json"))
print("Published Silver V2 quality reports:", available)
if SILVER_V2_REPORT_NAME is None:
    if len(available) != 1:
        raise ValueError(
            "Choose SILVER_V2_REPORT_NAME explicitly (one exact filename); "
            f"found {len(available)} reports: {available}"
        )
    SILVER_V2_REPORT_NAME = available[0]
if Path(SILVER_V2_REPORT_NAME).name != SILVER_V2_REPORT_NAME:
    raise ValueError("Silver report name must not contain directories")
silver_v2 = SilverReader(DATA_LAKE_ROOT, SILVER_V2_REPORT_NAME)
config = GoldConfig(seed=42)
pipeline = GoldV2Pipeline(silver_v2, config=config)
verified_silver_v2 = pipeline.verify_silver_publication()
print("Silver V2 + published artifacts + Silver V1 lineage VERIFIED:", verified_silver_v2)
print("Source:", silver_v2.report_path)
print("Cleaning ruleset:", silver_v2.report.get("ruleset_version"))
print("Accepted:", silver_v2.report["accepted_rows"])
print("Rejected:", silver_v2.report["rejected_rows"])


In [ ]:
# Output names are deterministic and isolated from Gold V1.
expected = pipeline.planned_paths(verified_silver_v2["sha256"])
existing = [str(path) for path in expected.values() if path.exists()]
stale_uploads = [
    str(path.with_name(path.name + ".uploading"))
    for path in expected.values()
    if path.with_name(path.name + ".uploading").exists()
]
print("Gold V2 planned output:", {name: str(path) for name, path in expected.items()})
print("Existing:", existing)
print("Unfinished uploads:", stale_uploads)
if stale_uploads:
    raise FileExistsError(f"Stale Gold V2 upload found: {stale_uploads}")
if existing and len(existing) != len(expected):
    raise FileExistsError(f"Partial Gold V2 publication found: {existing}")
if existing:
    gold_v2_reader = GoldReader(DATA_LAKE_ROOT, expected["manifest"].name)
    gold_verified = gold_v2_reader.verify()
    if (
        gold_v2_reader.manifest["source"]["silver_sha256"] != verified_silver_v2["sha256"]
        or gold_v2_reader.manifest["source"]["silver_report_sha256"]
        != pipeline._source_metadata(verified_silver_v2)["silver_report_sha256"]
    ):
        raise AssertionError("Existing Gold V2 lineage mismatch")
    print("Gold V2 already complete and VERIFIED:", gold_verified)


In [ ]:
PUBLISH_GOLD_V2 = False  # Opt in after review.
if not PUBLISH_GOLD_V2:
    print("Read-only preflight completed; set PUBLISH_GOLD_V2=True to publish.")
elif existing:
    print("Gold V2 exists and is verified; skipping write.")
else:
    result = pipeline.run()
    gold_v2_reader = GoldReader(DATA_LAKE_ROOT, expected["manifest"].name)
    print("GOLD V2 PUBLISHED:", result["paths"])
    print("VERIFIED:", gold_v2_reader.verify())


In [ ]:
# Optional read-only verification; does not expose held-out test to modeling.
if expected["manifest"].is_file():
    gold_v2_reader = GoldReader(DATA_LAKE_ROOT, expected["manifest"].name)
    validated = gold_v2_reader.verify()
    print("Gold V2 manifest:", gold_v2_reader.manifest_path)
    print("Splits:", validated["split_rows"])
    print("Version:", gold_v2_reader.manifest["pipeline_version"])
    print("Silver lineage:", gold_v2_reader.manifest["source"])
else:
    print("No complete Gold V2 publication yet. No data were modified.")


## Next step

After the published Gold V2 manifest is verified, train baselines separately using `scripts/06_train_baseline.py` with `--manifest-name` equal to the **Gold V2** manifest filename, and use a **new V2 experiment directory**. Choose models using train/validation only; do not inspect held-out test during selection. Source labels remain automatically assigned, not independently human verified.

Gold V2: `03_gold/v2/` and `04_reports/gold/v2/`. Gold V1 stays under `03_gold/` and `04_reports/` unchanged.
